In [ ]:
import numpy as np, pandas as pd, os, pickle, json
from pathlib import Path
from sklearn.metrics import mean_absolute_error
import warnings
import time
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

curr_path_file = Path("current_run.txt")
run_rel = curr_path_file.read_text().strip()
RUN_PATH = Path("experiments") / Path(run_rel)
RUN_PATH.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = RUN_PATH / "config.json"
CONFIG = json.load(open(CONFIG_PATH))
CACHE_DIR = RUN_PATH / "cache"

MODEL_NAME = "ARIMA"
PRED_DIR = RUN_PATH / "predictions" / MODEL_NAME
RESULTS_DIR = RUN_PATH / "results" / MODEL_NAME
MODELS_DIR = RUN_PATH / "models" / MODEL_NAME
for p in [PRED_DIR, RESULTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

all_tags = CONFIG["tags"]
horizons = CONFIG["horizons"]
region_targets = CONFIG["regions"]

tag = "P"
df_w = pd.read_csv(CONFIG["data_paths"]["price_series"], index_col="date", parse_dates=True)

print(f"Running ARIMA on {tag} tag × {len(region_targets)} regions × {len(horizons)} horizons")
print(f"Experiment: {CONFIG.get('exp_name','')} / {CONFIG.get('run_stamp','')}")

scores_mean_tag = {tag: {r: [np.inf] * len(horizons) for r in region_targets}}
scores_std_tag = {tag: {r: [0.0] * len(horizons) for r in region_targets}}

flat_npz_path = CACHE_DIR / "features_flat" / f"{tag}.npz"
folds_pkl_path = CACHE_DIR / "folds" / f"{tag}.pkl"

flat_npz = np.load(flat_npz_path, allow_pickle=False)
folds_map = pickle.load(open(folds_pkl_path, "rb"))

P_GRID = [0, 1, 2, 3]
D_GRID = [0, 1]
Q_GRID = [0, 1, 2, 3]

def _fmt_secs(x):
    x = int(max(0, x))
    m = x // 60
    s = x % 60
    h = m // 60
    m = m % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def _fit_arima_aic(train_values):
    y = np.asarray(train_values, dtype=float)
    y = y[np.isfinite(y)]

    best_order = None
    best_aic = np.inf

    for p in P_GRID:
        for d in D_GRID:
            for q in Q_GRID:
                if p == 0 and d == 0 and q == 0:
                    continue
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", category=UserWarning)
                    warnings.filterwarnings("ignore", category=RuntimeWarning)
                    warnings.filterwarnings("ignore", category=ConvergenceWarning)
                    fit = ARIMA(
                        y,
                        order=(p, d, q),
                        enforce_stationarity=True,
                        enforce_invertibility=True
                    ).fit()

                converged = True
                if hasattr(fit, "mle_retvals") and isinstance(fit.mle_retvals, dict):
                    converged = bool(fit.mle_retvals.get("converged", True))
                if not converged:
                    continue

                aic = float(fit.aic) if np.isfinite(fit.aic) else np.inf
                if aic < best_aic:
                    best_aic = aic
                    best_order = (p, d, q)

    return best_order, best_aic

def _fit_arima_fixed(values, order):
    y = np.asarray(values, dtype=float)
    y = y[np.isfinite(y)]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        fit = ARIMA(
            y,
            order=order,
            enforce_stationarity=True,
            enforce_invertibility=True
        ).fit()

    converged = True
    if hasattr(fit, "mle_retvals") and isinstance(fit.mle_retvals, dict):
        converged = bool(fit.mle_retvals.get("converged", True))
    if not converged:
        return None

    if not np.isfinite(getattr(fit, "aic", np.nan)):
        return None

    return fit

max_h = int(max(horizons)) if len(horizons) > 0 else 0

t0 = time.perf_counter()

for reg_i, reg in enumerate(region_targets):
    print(f"[Region {reg_i+1}/{len(region_targets)}] {reg}")

    series_full = df_w[reg].dropna()

    per_h_data = {}
    for h in horizons:
        kb = f"{reg}__h{h}"
        y_key = f"{kb}__y"
        idx_key = f"{kb}__idx"
        if y_key not in flat_npz or idx_key not in flat_npz:
            continue
        per_h_data[h] = {
            "y": flat_npz[y_key],
            "idx": pd.to_datetime(flat_npz[idx_key]),
            "folds": folds_map.get(kb, [])
        }


    fold_train_last_dates = []
    fold_key_set = set()
    for h in horizons:
        if h not in per_h_data:
            continue
        idx_h = per_h_data[h]["idx"]
        folds_h = per_h_data[h]["folds"]
        for fold_i, f in enumerate(folds_h):
            tr_mask, va_mask = f[0], f[1]
            train_last_date = pd.to_datetime(idx_h[tr_mask]).max()
            key = (fold_i, train_last_date)
            fold_key_set.add(key)
            fold_train_last_dates.append(train_last_date)

    fold_train_last_dates = sorted(set(fold_train_last_dates))

    print(f"Order selection: {len(fold_train_last_dates)} fold cutoffs")

    order_by_train_last = {}
    aic_by_train_last = {}

    sel_t0 = time.perf_counter()
    for j, train_last_date in enumerate(fold_train_last_dates):
        if train_last_date not in df_w.index:
            continue
        train_slice = series_full.loc[:train_last_date]
        order, aic = _fit_arima_aic(train_slice.values)
        if order is None:
            continue
        order_by_train_last[train_last_date] = order
        aic_by_train_last[train_last_date] = float(aic)
        if (j + 1) % 3 == 0 or (j + 1) == len(fold_train_last_dates):
            elapsed = time.perf_counter() - sel_t0
            print(f"Selected {j+1}/{len(fold_train_last_dates)}  |  elapsed {_fmt_secs(elapsed)}")


    for d in sorted(order_by_train_last.keys()):
        o = order_by_train_last[d]
        a = aic_by_train_last.get(d, np.nan)
        print(f"Fold cutoff {pd.to_datetime(d).date()}  ->  order {o}  |  AIC {a:.2f}")

    fit_ops_est = 0
    for h in horizons:
        if h not in per_h_data:
            continue
        idx_h = per_h_data[h]["idx"]
        folds_h = per_h_data[h]["folds"]
        for fold_i, f in enumerate(folds_h):
            tr_mask, va_mask = f[0], f[1]
            va_dates = pd.to_datetime(idx_h[va_mask])
            fit_ops_est += int(len(va_dates))

    print(f"Rolling refits (decision-time fits): est {fit_ops_est}")

    decision_fc_cache = {}
    done_fit_ops = 0
    tick_every_fits = 25

    maes_by_h = {h: [] for h in horizons}
    all_rows = []

    for h in horizons:
        if h not in per_h_data:
            continue

        yf = per_h_data[h]["y"]
        idx_h = per_h_data[h]["idx"]
        folds_h = per_h_data[h]["folds"]

        for fold_i, f in enumerate(folds_h):
            tr_mask, va_mask = f[0], f[1]

            train_last_date = pd.to_datetime(idx_h[tr_mask]).max()
            if train_last_date not in order_by_train_last:
                continue

            order = order_by_train_last[train_last_date]

            va_dates = pd.to_datetime(idx_h[va_mask])
            if len(va_dates) == 0:
                continue

            preds = np.empty(len(va_dates), dtype=float)
            preds[:] = np.nan

            for i_t, t_dec in enumerate(va_dates):
                target_date = pd.to_datetime(t_dec) + pd.DateOffset(weeks=int(h))
                if t_dec not in df_w.index or target_date not in df_w.index:
                    continue

                cache_key = (train_last_date, t_dec)
                if cache_key not in decision_fc_cache:
                    fit_start = time.perf_counter()

                    train_to_t = series_full.loc[:t_dec]
                    mdl = _fit_arima_fixed(train_to_t.values, order)
                    if mdl is None:
                        decision_fc_cache[cache_key] = None
                    else:
                        start_pos = int(df_w.index.get_loc(t_dec)) + 1
                        end_pos = int(df_w.index.get_loc(t_dec)) + int(max_h)
                        if end_pos >= len(df_w.index):
                            end_pos = len(df_w.index) - 1
                        steps = int(end_pos - start_pos + 1)
                        if steps <= 0:
                            decision_fc_cache[cache_key] = None
                        else:
                            try:
                                fc_vals = mdl.forecast(steps=steps)
                                fc_index = df_w.index[start_pos:end_pos + 1]
                                fc_series = pd.Series(np.asarray(fc_vals), index=fc_index)
                                decision_fc_cache[cache_key] = fc_series
                            except Exception:
                                decision_fc_cache[cache_key] = None

                    done_fit_ops += 1
                    if done_fit_ops % tick_every_fits == 0:
                        elapsed = time.perf_counter() - t0
                        avg = elapsed / max(1, done_fit_ops)
                        remaining = max(0, fit_ops_est - done_fit_ops) * avg
                        fit_elapsed = time.perf_counter() - fit_start
                        print(f"    Fits {done_fit_ops}/{fit_ops_est}  |  last {_fmt_secs(fit_elapsed)}  |  elapsed {_fmt_secs(elapsed)}  |  remaining est {_fmt_secs(remaining)}")

                fc_series = decision_fc_cache.get(cache_key, None)
                if fc_series is None:
                    continue

                v = fc_series.get(target_date, np.nan)
                if np.isfinite(v):
                    preds[i_t] = float(v)

            ok = np.isfinite(preds)
            if not np.all(ok):
                continue

            mae = mean_absolute_error(yf[va_mask], preds)
            maes_by_h[h].append(mae)

            all_rows.append(pd.DataFrame({
                "date": va_dates,
                "model": MODEL_NAME,
                "horizon": h,
                "fold": fold_i,
                "actual": yf[va_mask],
                "predicted": preds
            }))

    for hi, h in enumerate(horizons):
        vals = maes_by_h.get(h, [])
        if len(vals) > 0:
            scores_mean_tag[tag][reg][hi] = float(np.mean(vals))
            scores_std_tag[tag][reg][hi] = float(np.std(vals))
        else:
            scores_mean_tag[tag][reg][hi] = np.inf
            scores_std_tag[tag][reg][hi] = 0.0

    if all_rows:
        out_path = PRED_DIR / f"{tag}__{reg}__cv.parquet"
        pd.concat(all_rows, ignore_index=True).to_parquet(out_path, index=False)

rows = []
for reg in region_targets:
    for hi, h in enumerate(horizons):
        rows.append({
            "tag": tag,
            "region": reg,
            "model": MODEL_NAME,
            "horizon": h,
            "mae_mean": scores_mean_tag[tag][reg][hi],
            "mae_std": scores_std_tag[tag][reg][hi]
        })
pd.DataFrame(rows).to_csv(RESULTS_DIR / "scores_tag_all_cv.csv", index=False)

best = []
for reg in region_targets:
    for hi, h in enumerate(horizons):
        best.append({
            "region": reg,
            "model": MODEL_NAME,
            "horizon": h,
            "best_mae_mean": scores_mean_tag[tag][reg][hi]
        })
pd.DataFrame(best).to_csv(RESULTS_DIR / "scores_best_cv.csv", index=False)

with open(RESULTS_DIR / "scores_pickle_cv.pkl", "wb") as f:
    pickle.dump({"scores_mean_tag": scores_mean_tag, "scores_std_tag": scores_std_tag}, f)

elapsed_total = time.perf_counter() - t0
print(f"{MODEL_NAME} results saved under: {RUN_PATH}  | total elapsed {_fmt_secs(elapsed_total)}")


Running ARIMA on P tag × 19 regions × 9 horizons
Experiment: additive_tags_main_global_aug / 2026_01_26_123128
[Region 1/19] Business Bay
  Order selection: 10 fold cutoffs
    Selected 3/10  |  elapsed 18s
    Selected 6/10  |  elapsed 41s
    Selected 9/10  |  elapsed 1m 9s
    Selected 10/10  |  elapsed 1m 19s
    Fold cutoff 2020-08-30  ->  order (1, 0, 3)  |  AIC 629.52
    Fold cutoff 2021-02-28  ->  order (2, 1, 3)  |  AIC 682.19
    Fold cutoff 2021-08-29  ->  order (2, 1, 3)  |  AIC 768.33
    Fold cutoff 2022-02-27  ->  order (3, 0, 3)  |  AIC 841.13
    Fold cutoff 2022-08-28  ->  order (3, 0, 3)  |  AIC 948.88
    Fold cutoff 2023-02-26  ->  order (3, 0, 3)  |  AIC 1074.30
    Fold cutoff 2023-08-27  ->  order (3, 0, 3)  |  AIC 1140.86
    Fold cutoff 2024-02-25  ->  order (3, 0, 3)  |  AIC 1208.24
    Fold cutoff 2024-08-25  ->  order (3, 0, 3)  |  AIC 1280.67
    Fold cutoff 2025-02-23  ->  order (3, 0, 3)  |  AIC 1340.50
  Rolling refits (decision-time fits): est 2221
  